## Building a simple image-based LLM app 

References:
* https://github.com/openai/openai-python
* https://platform.openai.com/docs/api-reference/responses/create


In [38]:
from openai import OpenAI
import os

In [39]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

### Generate text from a simple prompt

In [40]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input = [
            {
                "role": "user",
                "content": "How do I check if a Python object is an instance of a class?"
            }
            ]
)

print(response.output_text)

In Python, you can check if an object is an instance of a particular class (or a subclass thereof) using the built-in function `isinstance()`.

### Syntax:
```python
isinstance(object, classinfo)
```

- `object`: the object you want to check.
- `classinfo`: a class, type, or a tuple of classes and types.

The function returns `True` if the object is an instance of the class or one of its subclasses, otherwise it returns `False`.

### Example:
```python
class Animal:
    pass

class Dog(Animal):
    pass

dog = Dog()

print(isinstance(dog, Dog))      # True
print(isinstance(dog, Animal))   # True, because Dog is a subclass of Animal
print(isinstance(dog, object))   # True, all classes inherit from object
print(isinstance(dog, list))     # False
```

### Checking Against Multiple Classes
You can also check against a tuple of classes:

```python
print(isinstance(dog, (Dog, list)))  # True because dog is a Dog
print(isinstance(dog, (int, list)))  # False
```

This is the most Pythonic and 

### Adding a system prompt prior to user input

In [41]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input = [
            {
                "role": "system",
                "content": "You are a coding assistant that talks like a pirate.",
            },
            {
                "role": "user",
                "content": "How do I check if a Python object is an instance of a class?"
            }
            ]
)

print(response.output_text)


Arrr, matey! To check if a Python object be an instance of a class, ye use the `isinstance()` function. It goes like this:

```python
isinstance(obj, ClassName)
```

If `obj` be an instance o' `ClassName` (or one o' its subclasses), it returns `True`; else it be `False`. 

Example, aye:

```python
class Pirate:
    pass

jack = Pirate()

print(isinstance(jack, Pirate))  # True, arrr!
```

That be how ye spy if an object be a member o' a class! Yo ho ho!


### Processing image inputs (Passing a URL)

In [42]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ]
)
print(response.output_text)


This image depicts a scenic landscape with a wooden boardwalk pathway running through a field of tall green grass. The sky above is blue with scattered white clouds, suggesting clear weather. In the distance, there are clusters of trees and shrubs along the horizon. The overall scene evokes a sense of tranquility and natural beauty, likely in a meadow, marsh, or nature reserve setting.


### Processing image inputs (passing a base64 encoded image)

In [43]:
import base64
# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
# Path to your image
image_path = "sample_image.jpg"

# Getting the Base64 string
encoded_string = encode_image(image_path)
image_url = f"data:image/png;base64,{encoded_string}"
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": image_url,
                    },
                ],
            }
    ]
)
print(response.output_text)

This image shows a wooden boardwalk path running through a lush green grassy field under a blue sky with scattered clouds. The path leads straight ahead, giving a sense of depth and inviting the viewer to walk forward. The grass on either side of the boardwalk is dense and vibrant, and there are some trees and shrubs visible in the distance. The lighting suggests that this picture might have been taken either in the morning or late afternoon, as the sunlight casts gentle shadows. The overall scene is peaceful and scenic, ideal for a nature walk or a quiet retreat.


### Process text and image inputs

In [44]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "what are the objects in this image",
                    },
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ]
)
print(response.output_text)


The image contains the following objects:
- A wooden pathway or boardwalk extending forward through the scene.
- Green grass and various plants surrounding the pathway.
- Trees and bushes in the distance.
- A partly cloudy blue sky above.


### Structured output
Specifying output format with json schema

In [45]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "what are the objects in this image",
                    },
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "object_description",
            "schema": {
                "type": "object",
                "properties": {
                    "objects": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "description": {"type": "string"},
                                "color": {"type": "string"},
                            },
                            "required": ["name", "description", "color"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["objects"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)
print(response.output_text)


{"objects":[{"name":"wooden pathway","description":"a wooden pathway cuts through the grass landscape","color":"light brown"},{"name":"grass","description":"tall green grass on either side of the wooden pathway","color":"green"},{"name":"trees","description":"clusters of trees visible in the background","color":"various shades of green"},{"name":"sky","description":"blue sky with scattered clouds","color":"blue and white"}]}


In [46]:
import json

def pretty_print_json(json_string):
    try:
        data = json.loads(json_string)
        print(json.dumps(data, indent=2))
    except json.JSONDecodeError as e:
        print("Invalid JSON:", e)
pretty_print_json(response.output_text)

{
  "objects": [
    {
      "name": "wooden pathway",
      "description": "a wooden pathway cuts through the grass landscape",
      "color": "light brown"
    },
    {
      "name": "grass",
      "description": "tall green grass on either side of the wooden pathway",
      "color": "green"
    },
    {
      "name": "trees",
      "description": "clusters of trees visible in the background",
      "color": "various shades of green"
    },
    {
      "name": "sky",
      "description": "blue sky with scattered clouds",
      "color": "blue and white"
    }
  ]
}


### Building Meal Lens
#### How It Works
1. **Image Upload**: Users can upload food photos via drag-and-drop or camera capture
2. **Base64 Encoding**: Images are converted to base64 format for API transmission
3. **AI Analysis**: OpenAI GPT-4 Vision API analyzes the image to extract:
   - Dish name and description
   - Ingredients list
   - Step-by-step recipe instructions
   - Nutritional information
   - Relevant tags and categories
   - Food pairing suggestions
#### Data Flow
1. User uploads image → Base64 conversion
2. Frontend sends to Next.js API route
3. API route forwards to FastAPI backend
4. FastAPI calls OpenAI GPT-4 Vision API
5. Structured JSON response returned through the chain
6. Frontend displays formatted recipe information

In [47]:
def analyze_food_image(base64_image: str):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": "You are a helpful culinary assistant that extracts structured recipe data from food images.  Make the output in a warm, wholesome tone like a southern grandma.",
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Extract the dish name, description of the dish, recipe, ingredients, nutrition facts, relevant tags, and suggested food pairings from this image. Keep the description short and sweet. Return the recipe in steps in markdown format (separate each step by a line break). Make the output in a warm, wholesome tone like a southern grandma.",
                    },
                    {
                        "type": "input_image",
                        "image_url": f"{base64_image}",
                    },
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "food_analysis",
                "schema": {
                    "type": "object",
                    "properties": {
                        "dish_name": {"type": "string"},
                        "description": {"type": "string"},
                        "tags": {"type": "array", "items": {"type": "string"}},
                        "recipe": {"type": "string"},
                        "ingredients": {"type": "array", "items": {"type": "string"}},
                        "nutrition_facts": {
                            "type": "object",
                            "properties": {
                                "serving_size": {"type": "string"},
                                "calories": {"type": "integer"},
                                "protein": {"type": "integer"},
                                "carbohydrates": {"type": "integer"},
                                "fat": {"type": "integer"},
                            },
                            "required": [
                                "serving_size",
                                "calories",
                                "protein",
                                "carbohydrates",
                                "fat",
                            ],
                            "additionalProperties": False,
                        },
                        "food_pairings": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                    },
                    "required": [
                        "dish_name",
                        "description",
                        "tags",
                        "recipe",
                        "ingredients",
                        "nutrition_facts",
                        "food_pairings",
                    ],  # Added to required
                    "additionalProperties": False,
                },
                "strict": True,
            }
        },
    )
    return response.output_text

In [48]:
image_url = f"data:image/png;base64,{encode_image("recipe_scallop.jpg")}"

In [49]:
response = analyze_food_image(image_url)

In [50]:
response

'{"dish_name":"Seared Scallops with Pea Puree and Crispy Bacon","description":"Deliciously seared scallops nestled on a bed of creamy pea puree, garnished with crispy bacon bits and fresh greens for a perfect bite.","tags":["seafood","appetizer","gluten-free","quick","elegant","spring"],"recipe":"1. Prepare the pea puree by cooking fresh peas until tender, then blending them with a bit of cream, salt, and pepper until smooth.\\n2. Pat the scallops dry and season them lightly with salt and pepper.\\n3. Heat a skillet with a touch of oil over medium-high heat.\\n4. Sear the scallops for about 2 minutes on each side until golden brown and just cooked through.\\n5. Cook bacon until crispy and chop into small pieces.\\n6. Spread the pea puree evenly on a serving plate.\\n7. Place the seared scallops gently on top of the puree.\\n8. Sprinkle the crispy bacon bits and some fresh microgreens or herbs over the dish.\\n9. Serve immediately and enjoy the fresh flavors!","ingredients":["Fresh scal

In [51]:
pretty_print_json(response)

{
  "dish_name": "Seared Scallops with Pea Puree and Crispy Bacon",
  "description": "Deliciously seared scallops nestled on a bed of creamy pea puree, garnished with crispy bacon bits and fresh greens for a perfect bite.",
  "tags": [
    "seafood",
    "appetizer",
    "gluten-free",
    "quick",
    "elegant",
    "spring"
  ],
  "recipe": "1. Prepare the pea puree by cooking fresh peas until tender, then blending them with a bit of cream, salt, and pepper until smooth.\n2. Pat the scallops dry and season them lightly with salt and pepper.\n3. Heat a skillet with a touch of oil over medium-high heat.\n4. Sear the scallops for about 2 minutes on each side until golden brown and just cooked through.\n5. Cook bacon until crispy and chop into small pieces.\n6. Spread the pea puree evenly on a serving plate.\n7. Place the seared scallops gently on top of the puree.\n8. Sprinkle the crispy bacon bits and some fresh microgreens or herbs over the dish.\n9. Serve immediately and enjoy the fr

### Building Deal to Meal
#### How It Works
1. **Banner Selection**: Users select from predefined grocery store banners (No Frills, Loblaws, T&T)
2. **Flyer URL Scraping**: System scrapes flyer urls from a public website
3. **AI Analysis**: OpenAI GPT-4 Vision API analyzes the flyer to extract all products on-sale and generates:
   - recipe using flyer products
   - estimated cost
   - step by step cooking instruction
   - nutrition analysis
#### Data Flow
1. User selects store banner
2. Frontend sends banner selection to API
3. Backend fetches flyer URLs for that store
4. GPT-4 Vision analyzes all flyer pages
5. AI generates recipe using available products
6. Returns structured recipe with cost and nutrition data

In [52]:
# Scrape latest flyer url for each banner from a public website
# For the purpose of demoing building an LLM app, this step is replaced by hard coding flyer urls here.

banner_flyer_dict= {
    "no_frills": ["https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-1.jpg",
                    "https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-2.jpg",
                    "https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-3.jpg"],
    "loblaws": ["https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-1.jpg",
                "https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-2.jpg",
                "https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-3.jpg"],
    "t_t": ["https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-1.jpg",
            "https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-2.jpg",
            "https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-3.jpg"]
            }
def generate_flyer_dinner(banner):
    urls = banner_flyer_dict[banner]
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Extract all products listed in these flyers. Then, generate one dinner recipe for two people that uses as many of those flyer products as possible. When referencing ingredients from the flyer, match their names exactly as shown. Finally, estimate the total cost of the dinner based on the flyer prices.",
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[0]
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[1]
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[2]
                    },              
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "flyer_dinner",
                "schema": {
                    "type": "object",
                    "properties": {
                        "dish_name": {"type": "string"},
                        "description": {"type": "string"},
                        "tags": {"type": "array", "items": {"type": "string"}},
                        "recipe": {"type": "string"},
                        "ingredients": {"type": "array", "items": {"type": "string"}},
                        "cost": {"type": "integer"},
                        "nutrition_facts": {
                            "type": "object",
                            "properties": {
                                "serving_size": {"type": "string"},
                                "calories": {"type": "integer"},
                                "protein": {"type": "integer"},
                                "carbohydrates": {"type": "integer"},
                                "fat": {"type": "integer"},
                            },
                            "required": [
                                "serving_size",
                                "calories",
                                "protein",
                                "carbohydrates",
                                "fat",
                            ],
                            "additionalProperties": False,
                        },
                    },
                    "required": [
                        "dish_name",
                        "description",
                        "tags",
                        "recipe",
                        "ingredients",
                        "nutrition_facts",
                        "cost",
                    ],  # Added to required
                    "additionalProperties": False,
                },
                "strict": True,
            }
        },
    )
    return {"llm_response": response.output_text,
            "urls":
            {"url1": urls[0],
            "url2": urls[1],
            "url3": urls[2]}}

In [53]:
response = generate_flyer_dinner("loblaws")
response

{'llm_response': '{"dish_name":"Salmon and Shrimp Stir Fry with Side Vegetables and Kettle Chips","description":"A delicious and easy stir fry for two, featuring Gold Seal Wild Sockeye Salmon and PC Pacific Large White Shrimp, sautéed with PC Vegetables (broccoli florets). Served with a side of honey dijon Kettle Chips and finished with PC Ketchup for added flavor.","tags":["seafood","stir fry","easy","dinner","quick","vegetables"],"recipe":"1. Thaw and pat dry the Gold Seal Wild Sockeye Salmon and PC Pacific Large White Shrimp.\\n2. In a skillet, heat some oil over medium heat. Add the broccoli florets from PC Vegetables and sauté until tender-crisp, about 5 minutes.\\n3. Add the salmon chunks and shrimp to the skillet. Cook until shrimp is pink and salmon is cooked through, about 4-5 minutes.\\n4. Drizzle PC Ketchup over the seafood and vegetables, stir to coat and heat through for 1-2 minutes.\\n5. Serve with a side of Honey Dijon Kettle Chips for a crunchy contrast.\\n6. Enjoy with

In [54]:
pretty_print_json(response["llm_response"])

{
  "dish_name": "Salmon and Shrimp Stir Fry with Side Vegetables and Kettle Chips",
  "description": "A delicious and easy stir fry for two, featuring Gold Seal Wild Sockeye Salmon and PC Pacific Large White Shrimp, saut\u00e9ed with PC Vegetables (broccoli florets). Served with a side of honey dijon Kettle Chips and finished with PC Ketchup for added flavor.",
  "tags": [
    "seafood",
    "stir fry",
    "easy",
    "dinner",
    "quick",
    "vegetables"
  ],
  "recipe": "1. Thaw and pat dry the Gold Seal Wild Sockeye Salmon and PC Pacific Large White Shrimp.\n2. In a skillet, heat some oil over medium heat. Add the broccoli florets from PC Vegetables and saut\u00e9 until tender-crisp, about 5 minutes.\n3. Add the salmon chunks and shrimp to the skillet. Cook until shrimp is pink and salmon is cooked through, about 4-5 minutes.\n4. Drizzle PC Ketchup over the seafood and vegetables, stir to coat and heat through for 1-2 minutes.\n5. Serve with a side of Honey Dijon Kettle Chips fo